# Low-dimensional Q70 alignment tuning and size-adjusted power (v5)

This notebook keeps the v4 **no-alignment** configuration unchanged and focuses on the Q70-aligned generator.

Observed v4 raw H0 rejection rates (0.10 / 0.05) were:

- no alignment: `0.17 / 0.06`;
- Q70 alignment with `lambda_align=0.05`: `0.18 / 0.10`.

The baseline is already close enough for the present tuning round, whereas the aligned method is still liberal. Therefore v5:

1. leaves all no-alignment learning parameters unchanged;
2. retunes only the aligned generator;
3. separates the two final H0 experiments, so the aligned H0 can be run alone;
4. screens `lambda_align` over `(0.01, 0.02, 0.03, 0.05, 0.10)`;
5. selects a provisional lambda by H0 size, then optionally checks size-adjusted power at `alpha_x=0.20`;
6. reruns the selected aligned configuration with 100 H0 repetitions before its full power curve.

The DGP, `n=400`, two-fold cross-fitting, `M_test=100`, and `n_boot=1000` are unchanged. The default manual fallback is `lambda_align=0.03`; the sweep result should take priority when available.

## 0. Setup

Keep this notebook and the modified `ci_test.py` in the same folder. The module must provide `size_adjusted_cutoffs` and accept `null_pvalues` in `run_experiment`.

The multi-GPU wrapper uses the device IDs listed in `GPU_IDS`.

In [ ]:
%pip install -q matplotlib pandas torch joblib

In [ ]:
from copy import deepcopy
import importlib
import inspect
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

import ci_test as C
C = importlib.reload(C)

assert hasattr(C, "size_adjusted_cutoffs"), (
    "The imported ci_test.py is old: size_adjusted_cutoffs is missing."
)
assert "null_pvalues" in inspect.signature(C.run_experiment).parameters, (
    "The imported ci_test.py is old: run_experiment has no null_pvalues argument."
)

print("ci_test loaded from:", Path(C.__file__).resolve())
print("Size-adjusted API check: PASSED")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

from joblib import Parallel, delayed

GPU_IDS = [0, 1, 2, 3]


def _split_seeds(n_rep, n_workers):
    seeds = list(range(int(n_rep)))
    n_workers = max(1, int(n_workers))
    base, rem = divmod(len(seeds), n_workers)
    chunks, start = [], 0
    for i in range(n_workers):
        size = base + (1 if i < rem else 0)
        chunks.append(seeds[start:start + size])
        start += size
    return chunks


def _register_q70_for_worker(ci_local):
    torch_local = ci_local.torch

    def levels(Z):
        hx = 0.8 + 0.7 * torch_local.sigmoid(
            0.8 * Z[:, [0]] - 0.6 * Z[:, [1]] + 0.4 * torch_local.sin(Z[:, [2]])
        )
        hy = 0.8 + 0.7 * torch_local.sigmoid(
            -0.7 * Z[:, [0]] + 0.5 * Z[:, [3]] + 0.3 * torch_local.cos(Z[:, [4]])
        )
        return hx, hy

    def inverse_cdf(u, h):
        c = 16.0 * h - 8.8
        left_tail = -c + (u / 0.05) * (c - 1.0)
        center_left = -1.0 + (u - 0.05) / 0.45
        center_right = h * (u - 0.50) / 0.20
        upper_tail = h + 0.2 * (u - 0.70) / 0.30
        return torch_local.where(
            u < 0.05,
            left_tail,
            torch_local.where(
                u < 0.50,
                center_left,
                torch_local.where(u < 0.70, center_right, upper_tail),
            ),
        )

    def checkerboard(n, tau, rho, device):
        if not 0.0 <= rho <= 1.0:
            raise ValueError("alpha_x must lie in [0, 1].")

        delta = rho * tau * (1.0 - tau)
        probs = torch_local.tensor(
            [
                tau * tau + delta,
                tau * (1.0 - tau) - delta,
                tau * (1.0 - tau) - delta,
                (1.0 - tau) ** 2 + delta,
            ],
            dtype=torch_local.float32,
            device=device,
        )

        cell = torch_local.multinomial(probs, n, replacement=True)
        x_high = (cell >= 2).reshape(-1, 1)
        y_high = ((cell == 1) | (cell == 3)).reshape(-1, 1)

        ux = torch_local.where(
            x_high,
            tau + (1.0 - tau) * torch_local.rand(n, 1, device=device),
            tau * torch_local.rand(n, 1, device=device),
        )
        uy = torch_local.where(
            y_high,
            tau + (1.0 - tau) * torch_local.rand(n, 1, device=device),
            tau * torch_local.rand(n, 1, device=device),
        )
        return ux, uy

    def sample(n, hypothesis="H0", device=None, dz=10, alpha_x=0.10, **_):
        device = device or ci_local.get_device(prefer_gpu=True)
        Z = torch_local.randn(n, int(dz), device=device)
        hx, hy = levels(Z)

        if hypothesis.upper() == "H0":
            ux = torch_local.rand(n, 1, device=device)
            uy = torch_local.rand(n, 1, device=device)
        elif hypothesis.upper() == "H1":
            ux, uy = checkerboard(n, 0.70, float(alpha_x), device)
        else:
            raise ValueError("hypothesis must be 'H0' or 'H1'.")

        return inverse_cdf(ux, hx), inverse_cdf(uy, hy), Z

    def oracle(Z, m, device=None, **_):
        device = device or Z.device
        Z = Z.to(device)
        hx, hy = levels(Z)
        ux = torch_local.rand(Z.shape[0], m, 1, device=device)
        uy = torch_local.rand(Z.shape[0], m, 1, device=device)
        return (
            inverse_cdf(ux, hx.unsqueeze(1)),
            inverse_cdf(uy, hy.unsqueeze(1)),
        )

    ci_local.register_dgp(
        ci_local.DGP(
            name="q70_lowdim",
            sample=sample,
            oracle=oracle,
            description="Worker-local low-dimensional Q70 DGP.",
        )
    )


def _multi_gpu_chunk_worker(gpu_id, seeds, params):
    import os
    os.environ["CUDA_VISIBLE_DEVICES"] = str(int(gpu_id))

    import numpy as np
    import ci_test as ci_local

    if params.get("register_q70", False):
        _register_q70_for_worker(ci_local)

    pvals = []
    for seed in seeds:
        pvals.append(
            ci_local._one_replicate(
                int(seed),
                params["n"],
                params["hypothesis"],
                params["config"],
                params["oracle"],
                params["data_kwargs"],
                True,
                params["dgp"],
            )
        )

    return list(seeds), np.asarray(pvals, dtype=float)


_ORIG_RUN_EXPERIMENT = C.run_experiment


def run_experiment(
    n=400,
    hypothesis="H0",
    n_rep=100,
    config=None,
    oracle=False,
    levels=(0.10, 0.05),
    data_kwargs=None,
    dgp="skew",
    n_jobs=1,
    prefer_gpu=True,
    verbose=True,
    null_pvalues=None,
    gpu_ids=None,
    **kwargs,
):
    data_kwargs = dict(data_kwargs or {})
    levels = tuple(float(x) for x in levels)
    ids = [int(g) for g in (GPU_IDS if gpu_ids is None else gpu_ids)]

    use_multi = bool(prefer_gpu) and torch.cuda.is_available() and len(ids) >= 1

    if not use_multi:
        return _ORIG_RUN_EXPERIMENT(
            n=n,
            hypothesis=hypothesis,
            n_rep=n_rep,
            config=config,
            oracle=oracle,
            levels=levels,
            data_kwargs=data_kwargs,
            dgp=dgp,
            n_jobs=n_jobs,
            prefer_gpu=prefer_gpu,
            verbose=verbose,
            null_pvalues=null_pvalues,
            **kwargs,
        )

    chunks = _split_seeds(n_rep, len(ids))
    params = {
        "n": n,
        "hypothesis": hypothesis,
        "config": config,
        "oracle": oracle,
        "data_kwargs": data_kwargs,
        "dgp": dgp,
        "register_q70": (dgp == "q70_lowdim"),
    }
    jobs = [(gid, chunk) for gid, chunk in zip(ids, chunks) if chunk]

    if verbose:
        print(
            f"[info] multi-GPU devices={[g for g, _ in jobs]} | reps={n_rep} | "
            f"split={[len(c) for _, c in jobs]}"
        )

    parts = Parallel(n_jobs=len(jobs), backend="loky", verbose=0)(
        delayed(_multi_gpu_chunk_worker)(gid, chunk, params)
        for gid, chunk in jobs
    )

    seed_to_pval = {}
    for seed_list, arr in parts:
        for seed, pval in zip(seed_list, arr):
            seed_to_pval[int(seed)] = float(pval)

    pvals = np.asarray(
        [seed_to_pval[s] for s in range(int(n_rep))],
        dtype=float,
    )

    rejection = {
        level: float(np.mean(pvals < level))
        for level in levels
    }

    if verbose:
        tag = "ORACLE" if oracle else f"depth={(config or {}).get('depth')}"
        print(
            f"[{hypothesis} dgp={dgp} {tag} n={n} reps={n_rep}] "
            + "  ".join(
                f"rej@{level:.2f}={rejection[level]:.3f}"
                for level in levels
            )
        )

    result = {
        "rejection": rejection,
        "pvalues": pvals,
    }

    if null_pvalues is not None:
        cutoffs = C.size_adjusted_cutoffs(null_pvalues, levels=levels)
        cutoffs = {
            float(k): float(v)
            for k, v in dict(cutoffs).items()
        }
        result["cutoffs"] = cutoffs
        result["size_adjusted_power"] = {
            level: float(np.mean(pvals <= cutoffs[level]))
            for level in levels
        }

    return result


C.run_experiment = run_experiment
print("Multi-GPU wrapper enabled. GPU_IDS =", GPU_IDS)


## 0.1 Register the low-dimensional Q70 DGP

In [ ]:
def _q70_levels(Z):
    if Z.ndim != 2 or Z.shape[1] < 5:
        raise ValueError("The Q70 DGP requires dz >= 5.")

    hx = 0.8 + 0.7 * torch.sigmoid(
        0.8 * Z[:, [0]]
        - 0.6 * Z[:, [1]]
        + 0.4 * torch.sin(Z[:, [2]])
    )
    hy = 0.8 + 0.7 * torch.sigmoid(
        -0.7 * Z[:, [0]]
        + 0.5 * Z[:, [3]]
        + 0.3 * torch.cos(Z[:, [4]])
    )
    return hx, hy


def _q70_inverse_cdf(u, h):
    c = 16.0 * h - 8.8
    left_tail = -c + (u / 0.05) * (c - 1.0)
    center_left = -1.0 + (u - 0.05) / 0.45
    center_right = h * (u - 0.50) / 0.20
    upper_tail = h + 0.2 * (u - 0.70) / 0.30

    return torch.where(
        u < 0.05,
        left_tail,
        torch.where(
            u < 0.50,
            center_left,
            torch.where(u < 0.70, center_right, upper_tail),
        ),
    )


def _q70_checkerboard_uniforms(n, tau, rho, device):
    if not 0.0 <= rho <= 1.0:
        raise ValueError("alpha_x must lie in [0, 1].")

    delta = rho * tau * (1.0 - tau)
    cell_probabilities = torch.tensor(
        [
            tau * tau + delta,
            tau * (1.0 - tau) - delta,
            tau * (1.0 - tau) - delta,
            (1.0 - tau) ** 2 + delta,
        ],
        dtype=torch.float32,
        device=device,
    )

    cell = torch.multinomial(cell_probabilities, n, replacement=True)

    x_high = (cell >= 2).reshape(-1, 1)
    y_high = ((cell == 1) | (cell == 3)).reshape(-1, 1)

    ux = torch.where(
        x_high,
        tau + (1.0 - tau) * torch.rand(n, 1, device=device),
        tau * torch.rand(n, 1, device=device),
    )
    uy = torch.where(
        y_high,
        tau + (1.0 - tau) * torch.rand(n, 1, device=device),
        tau * torch.rand(n, 1, device=device),
    )

    return ux, uy


def sample_q70_lowdim(
    n,
    hypothesis="H0",
    device=None,
    dz=10,
    alpha_x=0.10,
    **_,
):
    device = device or C.get_device(prefer_gpu=True)

    if int(dz) < 5:
        raise ValueError("Use dz >= 5 for the Q70 DGP.")

    Z = torch.randn(n, int(dz), device=device)
    hx, hy = _q70_levels(Z)

    if hypothesis.upper() == "H0":
        ux = torch.rand(n, 1, device=device)
        uy = torch.rand(n, 1, device=device)
    elif hypothesis.upper() == "H1":
        ux, uy = _q70_checkerboard_uniforms(
            n=n,
            tau=0.70,
            rho=float(alpha_x),
            device=device,
        )
    else:
        raise ValueError("hypothesis must be 'H0' or 'H1'.")

    X = _q70_inverse_cdf(ux, hx)
    Y = _q70_inverse_cdf(uy, hy)
    return X, Y, Z


def oracle_q70_lowdim(Z, m, device=None, **_):
    device = device or Z.device
    Z = Z.to(device)

    hx, hy = _q70_levels(Z)
    hx = hx.unsqueeze(1)
    hy = hy.unsqueeze(1)

    ux = torch.rand(Z.shape[0], m, 1, device=device)
    uy = torch.rand(Z.shape[0], m, 1, device=device)

    return (
        _q70_inverse_cdf(ux, hx),
        _q70_inverse_cdf(uy, hy),
    )


Q70_DGP = C.register_dgp(
    C.DGP(
        name="q70_lowdim",
        sample=sample_q70_lowdim,
        oracle=oracle_q70_lowdim,
        description=(
            "Low-dimensional Q70 DGP: dz=10; conditional means and medians are zero; "
            "Q0.70 varies nonlinearly with Z; H1 couples the Q70 exceedance cells."
        ),
    )
)

_X, _Y, _Z = Q70_DGP.sample(
    16,
    hypothesis="H0",
    device=torch.device("cpu"),
    dz=10,
)

assert _X.shape == _Y.shape == (16, 1)
assert _Z.shape == (16, 10)

print("Registered:", Q70_DGP.name)
print(Q70_DGP.description)


## 1. Global experiment and tuning settings

In [ ]:
RUN_PROFILE = "final"      # "quick" or "final"

if RUN_PROFILE == "quick":
    N_REP_ORACLE = 30
    N_REP_H0 = 50
    N_REP_H1 = 50
    N_REP_LAMBDA_H0 = 30
    N_REP_LAMBDA_POWER = 30
else:
    N_REP_ORACLE = 100
    N_REP_H0 = 100
    N_REP_H1 = 100
    N_REP_LAMBDA_H0 = 50
    N_REP_LAMBDA_POWER = 50

N = 400
LEVELS = (0.10, 0.05)
DGP_NAME = "q70_lowdim"
N_JOBS = -1
PREFER_GPU = True
BASE_DATA_KWARGS = {"dz": 10}

ALPHA_GRID = [
    0.05, 0.10, 0.15, 0.20,
    0.25, 0.30, 0.35, 0.40,
]

# Lambda screening. The same Monte Carlo seeds are reused for every candidate.
RUN_LAMBDA_SWEEP = True
RUN_LAMBDA_POWER_PILOT = True
LAMBDA_GRID = (0.01, 0.02, 0.03, 0.05, 0.10)
MANUAL_LAMBDA = 0.03
LAMBDA_SHORTLIST_N = 3
POWER_PILOT_ALPHA = 0.20

GPU_IDS = [0, 1, 2, 3]

print({
    "profile": RUN_PROFILE,
    "n": N,
    "final H0 repetitions": N_REP_H0,
    "lambda-screen H0 repetitions": N_REP_LAMBDA_H0,
    "lambda-power repetitions": N_REP_LAMBDA_POWER,
    "lambda grid": LAMBDA_GRID,
    "DGP": DGP_NAME,
    "GPU_IDS": GPU_IDS,
})

## 2. Inspect the registered DGP

In [ ]:
for name, dgp in C.DGPS.items():
    print(f"{name:16s}: {dgp.description}")

assert DGP_NAME in C.DGPS, f"Unknown DGP_NAME={DGP_NAME!r}"


## 3. Learning configurations

### Frozen no-alignment configuration

This is exactly the v4 baseline: `depth=3`, `noise_dim=8`, `lr=7.5e-4`, `batch_size=128`, `M_train=40`, `epochs=1200`, `min_epochs=180`, and `patience=140`.

### Retuned Q70-alignment template

Relative to the v4 aligned network, v5 uses:

- `lr: 5e-4 -> 4e-4`;
- `epochs: 1400 -> 1600`;
- `min_epochs: 220 -> 260`;
- `patience: 170 -> 200`;
- `lr_patience: 25 -> 30`;
- `align_samples: 192 -> 256`.

The three-layer architecture, batch size, `M_train=40`, kernel weights, and test-stage settings are retained. `make_q70_config(lambda_value)` changes only the alignment weight during the lambda sweep.

In [ ]:
# No alignment: frozen at the v4 settings.
CONFIG_L0 = dict(C.DEFAULT_CONFIG)
CONFIG_L0.update(
    depth=3,
    width=1024,
    noise_dim=8,
    dropout=0.0,
    lr=7.5e-4,
    epochs=1200,
    batch_size=128,
    grad_clip=None,
    weight_decay=1e-5,
    M_train=40,
    mmd_w_laplacian=1.0,
    mmd_w_gaussian=1.0,
    early_stop=True,
    min_epochs=180,
    patience=140,
    min_delta=1e-5,
    lr_scheduler=True,
    lr_factor=0.5,
    lr_patience=25,
    min_lr_frac=0.05,
    align_mode="none",
    lambda_align=0.0,
    taus=(0.70,),
    align_samples=64,
    n_folds=2,
    M_test=100,
    n_boot=1000,
    boot_rv="gaussian",
    standardize=True,
)


# Q70 alignment: v5 learning template. Only lambda_align varies in the sweep.
CONFIG_Q70_TEMPLATE = dict(C.DEFAULT_CONFIG)
CONFIG_Q70_TEMPLATE.update(
    depth=3,
    width=1024,
    noise_dim=8,
    dropout=0.0,
    lr=4.0e-4,
    epochs=1600,
    batch_size=128,
    grad_clip=None,
    weight_decay=1e-5,
    M_train=40,
    mmd_w_laplacian=1.0,
    mmd_w_gaussian=1.0,
    early_stop=True,
    min_epochs=260,
    patience=200,
    min_delta=1e-5,
    lr_scheduler=True,
    lr_factor=0.5,
    lr_patience=30,
    min_lr_frac=0.05,
    align_mode="quantile",
    lambda_align=MANUAL_LAMBDA,
    taus=(0.70,),
    align_samples=256,
    n_folds=2,
    M_test=100,
    n_boot=1000,
    boot_rv="gaussian",
    standardize=True,
)


def make_q70_config(lambda_value):
    cfg = deepcopy(CONFIG_Q70_TEMPLATE)
    cfg["lambda_align"] = float(lambda_value)
    return cfg


def lambda_tag(value):
    return f"{float(value):.3f}".rstrip("0").rstrip(".").replace(".", "p")


print("Frozen baseline lambda:", CONFIG_L0["lambda_align"])
print("Manual aligned fallback lambda:", MANUAL_LAMBDA)
print("Aligned template:", {
    key: CONFIG_Q70_TEMPLATE[key]
    for key in [
        "depth", "width", "noise_dim", "lr", "epochs", "batch_size",
        "M_train", "min_epochs", "patience", "min_delta", "lr_factor",
        "lr_patience", "min_lr_frac", "lambda_align", "taus",
        "align_samples", "n_folds", "M_test", "n_boot",
    ]
})

### 3.1 Verify that the lambda candidates differ only in `lambda_align`

In [ ]:
def config_difference_table(config_a, config_b):
    rows = []
    for key in sorted(set(config_a) | set(config_b)):
        if config_a.get(key) != config_b.get(key):
            rows.append({
                "parameter": key,
                "candidate_a": config_a.get(key),
                "candidate_b": config_b.get(key),
            })
    return pd.DataFrame(rows)


LAMBDA_ONLY_DIFFERENCE = config_difference_table(
    make_q70_config(LAMBDA_GRID[0]),
    make_q70_config(LAMBDA_GRID[-1]),
)
display(LAMBDA_ONLY_DIFFERENCE)
assert set(LAMBDA_ONLY_DIFFERENCE["parameter"]) == {"lambda_align"}

## 4. Optional oracle H0 sanity check

The oracle does not depend on learned-generator tuning. It may be skipped when it has already been checked for this DGP.

In [ ]:
ORACLE_RESULT = C.run_experiment(
    n=N,
    hypothesis="H0",
    n_rep=N_REP_ORACLE,
    config=CONFIG_L0,
    dgp=DGP_NAME,
    oracle=True,
    levels=LEVELS,
    data_kwargs=BASE_DATA_KWARGS,
    n_jobs=N_JOBS,
    prefer_gpu=PREFER_GPU,
    verbose=True,
)
ORACLE_RESULT

## 5. Lambda screening — aligned method only

This section does **not** run the no-alignment network. It uses 50 H0 replications per lambda in the final profile. These runs are a screening stage, not the final reported Type I error.

The same seeds are used across candidates, making their differences less affected by dataset-to-dataset randomness.

In [ ]:
LAMBDA_H0_RESULTS = {}

if RUN_LAMBDA_SWEEP:
    for lambda_value in LAMBDA_GRID:
        print()
        print("=" * 88)
        print(f"Aligned H0 lambda_align={lambda_value:.3f}")
        print("=" * 88)

        LAMBDA_H0_RESULTS[float(lambda_value)] = C.run_experiment(
            n=N,
            hypothesis="H0",
            n_rep=N_REP_LAMBDA_H0,
            config=make_q70_config(lambda_value),
            dgp=DGP_NAME,
            oracle=False,
            levels=LEVELS,
            data_kwargs=BASE_DATA_KWARGS,
            n_jobs=N_JOBS,
            prefer_gpu=PREFER_GPU,
            verbose=True,
        )
else:
    print("Lambda sweep skipped; MANUAL_LAMBDA will be used.")

## 6. Rank lambda values by H0 size

The score standardizes the deviations from 0.10 and 0.05 by their Monte Carlo standard errors. Smaller is better. The practical bands are computed from the actual number of replications rather than hard-coded for 100 runs.

In [ ]:
def mc_band(level, n_rep, z_value=1.96):
    se = np.sqrt(float(level) * (1.0 - float(level)) / int(n_rep))
    return max(0.0, float(level) - z_value * se), min(1.0, float(level) + z_value * se)


lambda_size_rows = []
for lambda_value, result in LAMBDA_H0_RESULTS.items():
    p0 = np.asarray(result["pvalues"], dtype=float)
    row = {"lambda_align": float(lambda_value), "H0_replications": len(p0)}
    score = 0.0
    all_in_band = True

    for level in LEVELS:
        level = float(level)
        rate = float(np.mean(p0 < level))
        se = np.sqrt(level * (1.0 - level) / len(p0))
        low, high = mc_band(level, len(p0))
        row[f"rej_{level:.2f}"] = rate
        row[f"band_low_{level:.2f}"] = low
        row[f"band_high_{level:.2f}"] = high
        row[f"in_band_{level:.2f}"] = low <= rate <= high
        score += ((rate - level) / se) ** 2
        all_in_band = all_in_band and (low <= rate <= high)

    row["both_levels_in_band"] = bool(all_in_band)
    row["size_score"] = float(score)
    lambda_size_rows.append(row)

LAMBDA_SIZE_TABLE = pd.DataFrame(lambda_size_rows)

if not LAMBDA_SIZE_TABLE.empty:
    LAMBDA_SIZE_TABLE = LAMBDA_SIZE_TABLE.sort_values(
        ["both_levels_in_band", "size_score", "lambda_align"],
        ascending=[False, True, True],
    ).reset_index(drop=True)
    display(LAMBDA_SIZE_TABLE)

    LAMBDA_SHORTLIST = LAMBDA_SIZE_TABLE.head(LAMBDA_SHORTLIST_N)[
        "lambda_align"
    ].astype(float).tolist()
    SIZE_BEST_LAMBDA = float(LAMBDA_SIZE_TABLE.iloc[0]["lambda_align"])
    print("H0-size shortlist:", LAMBDA_SHORTLIST)
    print("Best lambda by H0 size only:", SIZE_BEST_LAMBDA)
else:
    LAMBDA_SHORTLIST = [float(MANUAL_LAMBDA)]
    SIZE_BEST_LAMBDA = float(MANUAL_LAMBDA)
    print("No sweep results; using manual lambda:", SIZE_BEST_LAMBDA)

## 7. Optional power pilot for the H0-size shortlist

H0 size alone cannot define the overall best lambda: a very weak penalty may control size but add little power. This optional stage compares only the top H0 candidates at `alpha_x=0.20`, using each candidate's own pilot H0 p-values for size adjustment.

In [ ]:
LAMBDA_POWER_RESULTS = {}
lambda_power_rows = []

if RUN_LAMBDA_SWEEP and RUN_LAMBDA_POWER_PILOT:
    for lambda_value in LAMBDA_SHORTLIST:
        result = C.run_experiment(
            n=N,
            hypothesis="H1",
            n_rep=N_REP_LAMBDA_POWER,
            config=make_q70_config(lambda_value),
            dgp=DGP_NAME,
            oracle=False,
            levels=LEVELS,
            data_kwargs={
                **BASE_DATA_KWARGS,
                "alpha_x": float(POWER_PILOT_ALPHA),
            },
            n_jobs=N_JOBS,
            prefer_gpu=PREFER_GPU,
            verbose=False,
            null_pvalues=LAMBDA_H0_RESULTS[float(lambda_value)],
        )
        LAMBDA_POWER_RESULTS[float(lambda_value)] = result
        power_010 = float(result["size_adjusted_power"][0.10])
        power_005 = float(result["size_adjusted_power"][0.05])
        lambda_power_rows.append({
            "lambda_align": float(lambda_value),
            "pilot_alpha_x": float(POWER_PILOT_ALPHA),
            "adjusted_power_0.10": power_010,
            "adjusted_power_0.05": power_005,
            "mean_adjusted_power": 0.5 * (power_010 + power_005),
            "H1_replications": len(result["pvalues"]),
        })
        print(
            f"lambda={lambda_value:.3f}: "
            f"adjusted power@0.10={power_010:.3f}, "
            f"@0.05={power_005:.3f}"
        )
else:
    print("Power pilot skipped.")

LAMBDA_POWER_TABLE = pd.DataFrame(lambda_power_rows)
if not LAMBDA_POWER_TABLE.empty:
    display(LAMBDA_POWER_TABLE)

## 8. Choose the provisional lambda and construct the final aligned configuration

Selection rule:

1. require both H0 rejection rates to lie in their Monte Carlo bands when possible;
2. among eligible shortlisted values, prefer higher mean size-adjusted pilot power;
3. use the H0 size score to break ties;
4. if the sweep is skipped, use `MANUAL_LAMBDA=0.03`.

Because the screening experiments use only 50 replications, the selected value is approximate. The next aligned-only H0 cell performs the final 100-replication check.

In [ ]:
if not LAMBDA_POWER_TABLE.empty:
    selection_table = LAMBDA_SIZE_TABLE.merge(
        LAMBDA_POWER_TABLE,
        on="lambda_align",
        how="inner",
    )
    eligible = selection_table[selection_table["both_levels_in_band"]].copy()
    if eligible.empty:
        eligible = selection_table.copy()

    eligible = eligible.sort_values(
        ["mean_adjusted_power", "size_score", "lambda_align"],
        ascending=[False, True, True],
    )
    BEST_LAMBDA = float(eligible.iloc[0]["lambda_align"])
    display(selection_table.sort_values("lambda_align").reset_index(drop=True))
elif not LAMBDA_SIZE_TABLE.empty:
    BEST_LAMBDA = float(SIZE_BEST_LAMBDA)
else:
    BEST_LAMBDA = float(MANUAL_LAMBDA)

CONFIG_Q70_FINAL = make_q70_config(BEST_LAMBDA)

METHOD_LABELS = {
    "lambda_0": "No alignment (lambda=0)",
    "q70_aligned": f"Q70 alignment (tau=0.70, lambda={BEST_LAMBDA:g})",
}
METHOD_CONFIGS = {
    "lambda_0": CONFIG_L0,
    "q70_aligned": CONFIG_Q70_FINAL,
}

# The two H0 cells below populate this dictionary independently.
H0_RESULTS = {}
H1_RESULTS = {}

print("Selected provisional lambda_align =", BEST_LAMBDA)
print("Final aligned learning settings:", {
    key: CONFIG_Q70_FINAL[key]
    for key in [
        "depth", "width", "noise_dim", "lr", "epochs", "batch_size",
        "M_train", "min_epochs", "patience", "min_delta", "lr_factor",
        "lr_patience", "min_lr_frac", "lambda_align", "taus",
        "align_samples", "n_folds", "M_test", "n_boot",
    ]
})

## 9. Final learned-generator H0 experiments — separated

Run Section 9.2 alone if only the aligned method is needed. Section 9.1 is optional and does not have to be rerun.

### 9.1 Optional: no-alignment H0 only

In [ ]:
BASELINE_H0 = C.run_experiment(
    n=N,
    hypothesis="H0",
    n_rep=N_REP_H0,
    config=CONFIG_L0,
    dgp=DGP_NAME,
    oracle=False,
    levels=LEVELS,
    data_kwargs=BASE_DATA_KWARGS,
    n_jobs=N_JOBS,
    prefer_gpu=PREFER_GPU,
    verbose=True,
)
H0_RESULTS["lambda_0"] = BASELINE_H0

BASELINE_H0_FILE = Path("h0_q70_baseline_v5.npz")
np.savez_compressed(
    BASELINE_H0_FILE,
    pvalues=np.asarray(BASELINE_H0["pvalues"], dtype=float),
)
print("Saved baseline H0 to:", BASELINE_H0_FILE.resolve())

### 9.2 Q70-alignment H0 only

This is the main v5 experiment. It does not call the no-alignment configuration.

In [ ]:
ALIGNED_H0 = C.run_experiment(
    n=N,
    hypothesis="H0",
    n_rep=N_REP_H0,
    config=CONFIG_Q70_FINAL,
    dgp=DGP_NAME,
    oracle=False,
    levels=LEVELS,
    data_kwargs=BASE_DATA_KWARGS,
    n_jobs=N_JOBS,
    prefer_gpu=PREFER_GPU,
    verbose=True,
)
H0_RESULTS["q70_aligned"] = ALIGNED_H0

ALIGNED_H0_FILE = Path(
    f"h0_q70_aligned_lambda_{lambda_tag(BEST_LAMBDA)}_v5.npz"
)
np.savez_compressed(
    ALIGNED_H0_FILE,
    pvalues=np.asarray(ALIGNED_H0["pvalues"], dtype=float),
    lambda_align=np.asarray([BEST_LAMBDA], dtype=float),
)
print("Saved aligned H0 to:", ALIGNED_H0_FILE.resolve())

## 10. Inspect whichever final H0 experiments were run

In [ ]:
calibration_rows = []
ADJUSTED_CUTOFFS = {}
METHOD_SIZE_OK = {}

for method_id, h0_result in H0_RESULTS.items():
    p0 = np.asarray(h0_result["pvalues"], dtype=float)
    cutoffs = C.size_adjusted_cutoffs(p0, levels=LEVELS)
    ADJUSTED_CUTOFFS[method_id] = cutoffs
    method_ok = True

    for level in LEVELS:
        level = float(level)
        rate = float(np.mean(p0 < level))
        low, high = mc_band(level, len(p0))
        in_band = low <= rate <= high
        method_ok = method_ok and in_band
        calibration_rows.append({
            "method_id": method_id,
            "method": METHOD_LABELS[method_id],
            "nominal_level": level,
            "raw_H0_rejection": rate,
            "normal_band_low": low,
            "normal_band_high": high,
            "in_band": in_band,
            "adjusted_cutoff": float(cutoffs[level]),
            "H0_rejection_at_adjusted_cutoff": float(
                np.mean(p0 <= float(cutoffs[level]))
            ),
            "H0_replications": len(p0),
        })

    METHOD_SIZE_OK[method_id] = bool(method_ok)

CALIBRATION_TABLE = pd.DataFrame(calibration_rows)
if CALIBRATION_TABLE.empty:
    print("No final H0 result is available. Run Section 9.1 and/or 9.2.")
else:
    display(CALIBRATION_TABLE)
    print("Per-method size diagnostic:", METHOD_SIZE_OK)

### 10.1 Optional reload after a kernel restart

Set the flags for the files that exist. Loading aligned H0 does not require loading baseline H0.

In [ ]:
LOAD_BASELINE_H0 = False
LOAD_ALIGNED_H0 = True

H0_RESULTS = {}

if LOAD_BASELINE_H0:
    baseline_file = Path("h0_q70_baseline_v5.npz")
    baseline_loaded = np.load(baseline_file)
    p0 = np.asarray(baseline_loaded["pvalues"], dtype=float)
    H0_RESULTS["lambda_0"] = {
        "pvalues": p0,
        "rejection": {
            float(level): float(np.mean(p0 < float(level)))
            for level in LEVELS
        },
    }
    print("Loaded:", baseline_file.resolve())

if LOAD_ALIGNED_H0:
    aligned_file = Path(
        f"h0_q70_aligned_lambda_{lambda_tag(BEST_LAMBDA)}_v5.npz"
    )
    aligned_loaded = np.load(aligned_file)
    stored_lambda = float(np.asarray(aligned_loaded["lambda_align"]).reshape(-1)[0])
    if not np.isclose(stored_lambda, BEST_LAMBDA):
        raise ValueError(
            f"Stored lambda {stored_lambda} does not match BEST_LAMBDA {BEST_LAMBDA}."
        )
    p0 = np.asarray(aligned_loaded["pvalues"], dtype=float)
    H0_RESULTS["q70_aligned"] = {
        "pvalues": p0,
        "rejection": {
            float(level): float(np.mean(p0 < float(level)))
            for level in LEVELS
        },
    }
    print("Loaded:", aligned_file.resolve())

print("Available final H0 methods:", list(H0_RESULTS))

## 11. Q70-alignment full size-adjusted power — independent

This cell requires only the aligned H0 result from Section 9.2 (or the aligned reload). It does not require baseline H0.

In [ ]:
if "q70_aligned" not in H0_RESULTS:
    raise RuntimeError("Run or reload the aligned H0 result before aligned power.")

aligned_p0 = np.asarray(H0_RESULTS["q70_aligned"]["pvalues"], dtype=float)
aligned_size_ok = all(
    mc_band(float(level), len(aligned_p0))[0]
    <= float(np.mean(aligned_p0 < float(level)))
    <= mc_band(float(level), len(aligned_p0))[1]
    for level in LEVELS
)

if not aligned_size_ok:
    print(
        "WARNING: the final aligned H0 result is outside its Monte Carlo band. "
        "Treat the power curve as exploratory until the aligned size is acceptable."
    )

aligned_power_rows = []
H1_RESULTS["q70_aligned"] = {}

for alpha_x in ALPHA_GRID:
    result = C.run_experiment(
        n=N,
        hypothesis="H1",
        n_rep=N_REP_H1,
        config=CONFIG_Q70_FINAL,
        dgp=DGP_NAME,
        oracle=False,
        levels=LEVELS,
        data_kwargs={**BASE_DATA_KWARGS, "alpha_x": float(alpha_x)},
        n_jobs=N_JOBS,
        prefer_gpu=PREFER_GPU,
        verbose=False,
        null_pvalues=H0_RESULTS["q70_aligned"],
    )
    H1_RESULTS["q70_aligned"][float(alpha_x)] = result
    row = {
        "method_id": "q70_aligned",
        "method": METHOD_LABELS["q70_aligned"],
        "lambda_align": BEST_LAMBDA,
        "alpha_x": float(alpha_x),
        "size_adjusted_power_0.10": result["size_adjusted_power"][0.10],
        "size_adjusted_power_0.05": result["size_adjusted_power"][0.05],
        "cutoff_0.10": result["cutoffs"][0.10],
        "cutoff_0.05": result["cutoffs"][0.05],
        "H1_replications": len(result["pvalues"]),
    }
    aligned_power_rows.append(row)
    print(
        f"alpha_x={alpha_x:.2f}: adjusted power@0.10="
        f"{row['size_adjusted_power_0.10']:.3f}, @0.05="
        f"{row['size_adjusted_power_0.05']:.3f}"
    )

ALIGNED_POWER_TABLE = pd.DataFrame(aligned_power_rows)
display(ALIGNED_POWER_TABLE)

## 12. Optional no-alignment full power

Run this only when a fresh baseline comparison is required.

In [ ]:
if "lambda_0" not in H0_RESULTS:
    raise RuntimeError("Run or reload baseline H0 before baseline power.")

baseline_power_rows = []
H1_RESULTS["lambda_0"] = {}

for alpha_x in ALPHA_GRID:
    result = C.run_experiment(
        n=N,
        hypothesis="H1",
        n_rep=N_REP_H1,
        config=CONFIG_L0,
        dgp=DGP_NAME,
        oracle=False,
        levels=LEVELS,
        data_kwargs={**BASE_DATA_KWARGS, "alpha_x": float(alpha_x)},
        n_jobs=N_JOBS,
        prefer_gpu=PREFER_GPU,
        verbose=False,
        null_pvalues=H0_RESULTS["lambda_0"],
    )
    H1_RESULTS["lambda_0"][float(alpha_x)] = result
    baseline_power_rows.append({
        "method_id": "lambda_0",
        "method": METHOD_LABELS["lambda_0"],
        "lambda_align": 0.0,
        "alpha_x": float(alpha_x),
        "size_adjusted_power_0.10": result["size_adjusted_power"][0.10],
        "size_adjusted_power_0.05": result["size_adjusted_power"][0.05],
        "cutoff_0.10": result["cutoffs"][0.10],
        "cutoff_0.05": result["cutoffs"][0.05],
        "H1_replications": len(result["pvalues"]),
    })

BASELINE_POWER_TABLE = pd.DataFrame(baseline_power_rows)
display(BASELINE_POWER_TABLE)

## 13. Available comparison tables and power curves

In [ ]:
available_power_tables = []
if "ALIGNED_POWER_TABLE" in globals() and not ALIGNED_POWER_TABLE.empty:
    available_power_tables.append(ALIGNED_POWER_TABLE)
if "BASELINE_POWER_TABLE" in globals() and not BASELINE_POWER_TABLE.empty:
    available_power_tables.append(BASELINE_POWER_TABLE)

if not available_power_tables:
    print("No full power result is available.")
else:
    POWER_TABLE = pd.concat(available_power_tables, ignore_index=True)
    display(POWER_TABLE)

    for level, column in [
        (0.10, "size_adjusted_power_0.10"),
        (0.05, "size_adjusted_power_0.05"),
    ]:
        plt.figure(figsize=(8, 5))
        for method_id, group in POWER_TABLE.groupby("method_id"):
            group = group.sort_values("alpha_x")
            plt.plot(
                group["alpha_x"],
                group[column],
                marker="o",
                linewidth=2,
                label=METHOD_LABELS[method_id],
            )
        plt.xlabel("Dependence strength (alpha_x)")
        plt.ylabel(f"Size-adjusted power at level {level:.2f}")
        plt.ylim(0.0, 1.05)
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.show()

## 14. Save available v5 tuning and power results

In [ ]:
OUTPUT_DIR = Path("q70_quantile_v5_results")
OUTPUT_DIR.mkdir(exist_ok=True)

if not LAMBDA_SIZE_TABLE.empty:
    LAMBDA_SIZE_TABLE.to_csv(OUTPUT_DIR / "lambda_h0_screen.csv", index=False)
if not LAMBDA_POWER_TABLE.empty:
    LAMBDA_POWER_TABLE.to_csv(OUTPUT_DIR / "lambda_power_pilot.csv", index=False)
if "CALIBRATION_TABLE" in globals() and not CALIBRATION_TABLE.empty:
    CALIBRATION_TABLE.to_csv(OUTPUT_DIR / "final_h0_summary.csv", index=False)
if "ALIGNED_POWER_TABLE" in globals() and not ALIGNED_POWER_TABLE.empty:
    ALIGNED_POWER_TABLE.to_csv(OUTPUT_DIR / "aligned_power.csv", index=False)
if "BASELINE_POWER_TABLE" in globals() and not BASELINE_POWER_TABLE.empty:
    BASELINE_POWER_TABLE.to_csv(OUTPUT_DIR / "baseline_power.csv", index=False)
if "POWER_TABLE" in globals() and not POWER_TABLE.empty:
    POWER_TABLE.to_csv(OUTPUT_DIR / "available_power_long.csv", index=False)

np.savez_compressed(
    OUTPUT_DIR / "lambda_screen_h0_pvalues.npz",
    **{
        f"lambda_{lambda_tag(value)}": np.asarray(result["pvalues"], dtype=float)
        for value, result in LAMBDA_H0_RESULTS.items()
    },
)

print("Selected lambda_align:", BEST_LAMBDA)
print("Saved available v5 results to:", OUTPUT_DIR.resolve())